In [56]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# 1. Load dataset

In [57]:
try:
    df = pd.read_csv('trader_behaviour4.csv')
    print("Success: dataset loaded.")
except FileNotFoundError:
    print("Error: trader_behaviour4.csv not found.")
    exit()

Success: dataset loaded.


# 2. Select the 5 behavioral features

In [58]:

features = [
    'time_delta_seconds',
    'lot_size_multiplier',
    'is_unhedged',
    'loss_streak_count',
    'recent_pnl_delta'
]

X = df[features]
y = df['label']



# 3. Train/test split

In [59]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(
    f"Data split complete. "
    f"Training: {X_train.shape[0]} rows | "
    f"Testing: {X_test.shape[0]} rows"
)

Data split complete. Training: 4000 rows | Testing: 1000 rows


# 4. Train Random Forest

In [60]:
print("Training Random Forest...")

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    class_weight='balanced',
    random_state=42
)

model.fit(X_train, y_train)

Training Random Forest...


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

# 5. Evaluate on held-out test set

In [61]:
y_pred = model.predict(X_test)

print("\nHeld-Out Test Set Metrics:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Normal", "Potential Revenge"]
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Held-Out Test Set Metrics:
                   precision    recall  f1-score   support

           Normal       0.98      0.98      0.98       883
Potential Revenge       0.82      0.81      0.82       117

         accuracy                           0.96      1000
        macro avg       0.90      0.89      0.90      1000
     weighted avg       0.96      0.96      0.96      1000

Confusion Matrix:
[[862  21]
 [ 22  95]]


# 6. Save model

In [ ]:

# joblib.dump(model, 'model2.pkl')

# print("\nModel saved as model.pkl")


Model saved as model.pkl


In [63]:
print("Train accuracy:", model.score(X_train, y_train))
print("Test accuracy:", model.score(X_test, y_test))

Train accuracy: 0.976
Test accuracy: 0.957


In [64]:
from sklearn.metrics import recall_score

revenge_recall = recall_score(y_test, y_pred)

print(f"Potential Revenge Recall: {revenge_recall:.2f}")

Potential Revenge Recall: 0.81


In [65]:
fn_mask = (y_test == 1) & (y_pred == 0)

false_negatives = X_test[fn_mask]

print("False negatives:", len(false_negatives))
print(false_negatives.describe())

False negatives: 22
       time_delta_seconds  lot_size_multiplier  is_unhedged  \
count           22.000000            22.000000    22.000000   
mean            40.108138             1.457472     0.590909   
std             37.946720             0.550246     0.503236   
min              0.132139             0.855087     0.000000   
25%             13.125456             1.077950     0.000000   
50%             26.919096             1.295745     1.000000   
75%             57.162151             1.696312     1.000000   
max            149.949122             3.034485     1.000000   

       loss_streak_count  recent_pnl_delta  
count          22.000000         22.000000  
mean            0.909091      -1155.212633  
std             1.191601       3435.616609  
min             0.000000      -6895.161896  
25%             0.000000      -4018.829627  
50%             0.000000       -487.407607  
75%             1.750000       2025.161633  
max             3.000000       3813.653399  


In [66]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       883
           1       0.82      0.81      0.82       117

    accuracy                           0.96      1000
   macro avg       0.90      0.89      0.90      1000
weighted avg       0.96      0.96      0.96      1000

[[862  21]
 [ 22  95]]


In [67]:
print(df.groupby("label")[features].mean())

       time_delta_seconds  lot_size_multiplier  is_unhedged  \
label                                                         
0               45.478244             1.436744     0.511433   
1               21.125302             2.314214     0.864494   

       loss_streak_count  recent_pnl_delta  
label                                       
0               0.945891        -899.70418  
1               2.564322       -3420.93249  


In [68]:
print(df.isnull().sum())
print(df.describe())

Unnamed: 0             0
time_delta_seconds     0
lot_size_multiplier    0
is_unhedged            0
loss_streak_count      0
recent_pnl_delta       0
label                  0
dtype: int64
        Unnamed: 0  time_delta_seconds  lot_size_multiplier  is_unhedged  \
count  5000.000000         5000.000000          5000.000000  5000.000000   
mean   2499.500000           42.638691             1.539057     0.552600   
std    1443.520003           46.955893             0.600661     0.497275   
min       0.000000            0.000801             0.800016     0.000000   
25%    1249.750000           10.568512             1.111153     0.000000   
50%    2499.500000           26.697911             1.417500     1.000000   
75%    3749.250000           58.430709             1.717384     1.000000   
max    4999.000000          403.402481             3.499578     1.000000   

       loss_streak_count  recent_pnl_delta        label  
count        5000.000000       5000.000000  5000.000000  
mean       

In [70]:
for feature, importance in zip(features, model.feature_importances_):
    print(f"{feature}: {importance:.3f}")

time_delta_seconds: 0.099
lot_size_multiplier: 0.291
is_unhedged: 0.105
loss_streak_count: 0.333
recent_pnl_delta: 0.172


In [72]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Probabilities for class 1 (Potential Revenge) on held-out test set
y_probs = model.predict_proba(X_test)[:, 1]

thresholds = [0.35, 0.50, 0.55, 0.60, 0.65, 0.70]
results = []

for t in thresholds:
    y_pred_t = (y_probs >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    tn, fp, fn, tp = cm.ravel()
    
    results.append({
        "Threshold": t,
        "Precision": round(precision_score(y_test, y_pred_t, zero_division=0), 3),
        "Recall": round(recall_score(y_test, y_pred_t, zero_division=0), 3),
        "F1-Score": round(f1_score(y_test, y_pred_t, zero_division=0), 3),
        "False Positives (Blocked incorrectly)": fp,
        "False Negatives (Missed revenge)": fn,
        "True Positives (Revenge caught)": tp,
    })

eval_df = pd.DataFrame(results)
print(eval_df.to_string(index=False))

 Threshold  Precision  Recall  F1-Score  False Positives (Blocked incorrectly)  False Negatives (Missed revenge)  True Positives (Revenge caught)
      0.35      0.698   0.829     0.758                                     42                                20                               97
      0.50      0.819   0.812     0.815                                     21                                22                               95
      0.55      0.845   0.795     0.819                                     17                                24                               93
      0.60      0.861   0.795     0.827                                     15                                24                               93
      0.65      0.883   0.778     0.827                                     12                                26                               91
      0.70      0.936   0.752     0.834                                      6                                29            